# 41 · Widgets, job parameters and task values

**Scenario:** Pass a validated date from a job/widget into SQL and return a compact task result.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Default the date from the actual asset

A data-derived default avoids relying on today’s date matching a historical sample. Job parameters can override the widget.


In [ ]:
from datetime import date
default_date=df.agg(F.min('pickup_date')).first()[0]
assert default_date is not None
dbutils.widgets.text('run_date',default_date.isoformat())
run_date=date.fromisoformat(dbutils.widgets.get('run_date'))
df.createOrReplaceTempView('job_taxi_asset')
result=spark.sql('SELECT * FROM job_taxi_asset WHERE pickup_date=:run_date',args={'run_date':run_date})
assert result.filter(F.col('pickup_date')!=F.lit(run_date)).count()==0
print('Job date:',run_date,'rows:',result.count())


**Expected check:** The parsed date is bound as a value; a date outside the sample can legitimately return zero rows.


## Step 3 — Publish task values only in a job run

Notebook variables do not pass between tasks. The example is opt-in because taskValues.set belongs to a Jobs execution context.


In [ ]:
PUBLISH_JOB_VALUES=False
manifest={'source_asset':SOURCE_ASSET,'run_date':run_date.isoformat(),'rows':result.count()}
if PUBLISH_JOB_VALUES:
    dbutils.jobs.taskValues.set(key='source_manifest',value=manifest)
else:
    print('Interactive manifest:',manifest)
print('In a downstream task, use {{tasks.<task_key>.values.source_manifest}} or taskValues.get with a known task key.')


**Expected check:** Default interactive mode prints a manifest without requiring a job run.


## Practice

Store a SQL template in a repository .sql file and pass run_date from a job parameter into that template.

## Solution guide

Read the trusted template text and call spark.sql(template,args={"run_date":run_date}). For identifiers, use IDENTIFIER with args on a supported runtime, not raw substitution.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
